# Six-month future billed value

## 01. Executive summary

Forecast gross issued GBP for a fictional canonical account over six calendar
months. The experiment compares a training median with three regression families
under the frozen temporal and customer-isolated design. This is a bounded
billed-value proxy. Profit, collected cash and business readiness require
separate evidence.

## 02. Business and economic target

The approved target is `future_6m_issued_gbp` in GBP over (t, t+6 calendar months].
There is no discount, margin or lifetime extrapolation. A recorded advance issue
contributes in its issue month, even when it covers future service.

## 03. Customer-level prediction entity

The customer is the modelling entity. Each invoice belongs to one canonical
episode, and legitimately attributed successor or concurrent episodes can
contribute distinct future invoices. Accounts enter at known creation. No active
episode is required. The account and timestamp keys remain outside predictors.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from matplotlib.ticker import StrMethodFormatter

from nowa_customer_intelligence.models.clv.research import WORKSPACE, load_report

project = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
workspace = Path(os.environ.get("NCI_CLV_WORKSPACE", project / WORKSPACE))
report = load_report(workspace)
design = report["design"]

plt.style.use("seaborn-v0_8-whitegrid")
blue, green, grey = "#245584", "#148279", "#65778b"
data = pd.DataFrame(report["validation"])
data["Error GBP"] = data["Predicted GBP"] - data["Actual GBP"]
data["Absolute error GBP"] = data["Error GBP"].abs()
scores = pd.DataFrame({name: values["validation"] for name, values in report["scores"].items()}).T
selected = report["frozen"]["name"]


def finish():
    plt.tight_layout()
    plt.show()
    plt.close("all")


def currency(axis):
    axis.set_major_formatter(StrMethodFormatter("£{x:,.0f}"))


# Recover group keys from the verified development partition, never from reserve data.
if "partition_sha256" in design:
    from nowa_customer_intelligence.models.clv.design import assemble
    from nowa_customer_intelligence.models.clv.reference import load_reference_history
    from nowa_customer_intelligence.models.clv.research import _fingerprint

    ledger, _ = load_reference_history()
    partitions, audit = assemble(ledger, complete_through=ledger.cutoff)
    validation = partitions["validation"]
    assert audit == design["development_audit"]
    assert _fingerprint(validation) == design["partition_sha256"]["validation"]
    assert len(validation) == len(data)
    assert all(
        float(row.target.future_6m_issued_gbp) == evidence["Actual GBP"]
        and row.predictors["cadence"] == evidence["Cadence"]
        and row.target.observed_at.strftime("%Y-%m") == evidence["Observation month"]
        and float(row.predictors["issued_gbp"]) == evidence["Historical GBP"]
        and row.predictors["tenure_days"] == evidence["Tenure days"]
        for row, evidence in zip(validation, report["validation"], strict=True)
    )
    data["Account key"] = [row.target.customer_id for row in validation]
    assert data["Account key"].nunique() == design["support"]["validation"]["accounts"]
else:
    # Minimal test reports have no recoverable account identifiers.
    data["Account key"] = pd.NA

### Executive model dashboard

The Project Owner accepted the engineering workflow with documented limitations. The approved forecast is a six-month gross issued GBP proxy. Commercial readiness requires separate evidence.

In [ ]:
headline = pd.DataFrame(design["support"]).T
base_mae = scores.at["Training median", "MAE"]
chosen_mae = scores.at[selected, "MAE"]
denominators = [col for col in ("observations", "accounts") if col in headline]
display(headline[denominators])
display(
    pd.DataFrame(
        [
            ("Target", "Six-calendar-month gross issued GBP"),
            ("Raw features", len(design["features"])),
            ("Candidates", ", ".join(scores.index)),
            ("Frozen nominee", selected),
            ("Nominee validation MAE", f"£{chosen_mae:,.2f}"),
            ("Median benchmark MAE", f"£{base_mae:,.2f}"),
            (
                "Relative MAE reduction",
                f"{1 - chosen_mae / base_mae:.2%}" if base_mae else "Undefined",
            ),
            ("Reserve MAE", f"£{report['reserve']['MAE']:,.2f}"),
        ],
        columns=["Indicator", "Verified evidence"],
    )
)

## 04. Modelling population

Review distinct accounts alongside observations. Development contains repeated
account snapshots. Each account and its episodes belong to one development
partition. Reserve identifiers use a separately generated namespace and Registry.

## 05. Target maturity

A mature zero needs complete issuance evidence, including late availability.
In this verified fictional scenario, deterministic scheduling and the export
gates attest to the generated ledger through extraction. Unmatured or incomplete
targets remain missing. These are counts before group and window selection.

The reference validation has 355 observations but only 180 accounts. A row is
one forecast opportunity. Repeat observations contribute to row-level losses,
while uncertainty resamples accounts as blocks.

In [ ]:
display(pd.DataFrame(design["support"]).T)
display(
    pd.DataFrame(
        {
            "Seed 42": design["development_audit"]["states"],
            "Seed 731": design["reserve_audit"]["states"],
        }
    )
    .fillna(0)
    .astype(int)
)
display(
    pd.Series(design["development_audit"]["selection_exclusions"], name="Development exclusions")
)

### Target maturity before partitioning

Each row is a customer-month observation. The roster includes repeated accounts. A mature zero is a verified empty issue window. An immature or incomplete row has an unknown outcome. Calendar and group exclusions reduce the supervised roster further.

In [ ]:
state_counts = pd.DataFrame(
    {
        "Development, seed 42": design["development_audit"]["states"],
        "Reserve, seed 731": design["reserve_audit"]["states"],
    }
).fillna(0)
display(state_counts.astype(int))
ax = state_counts.T.plot.barh(stacked=True, figsize=(8, 3))
ax.set(
    title="Outcome evidence before partition selection", xlabel="Customer observations", ylabel=""
)
ax.legend(title="Target state", loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=3)
finish()

## 06. Financial target distribution

The validation target distribution combines literal confirmed zeros and positive
issuance. Inspect the upper tail with a sample count and quantiles before comparing
error. The histogram is based on mature development observations.

## 07. Billing frequency effects

Annual billing can place a full charge or no charge in the horizon. Cadence
in the table uses historical evidence. Mixed cadence can follow reactivation,
and an unobserved cadence remains a distinct category.

The upper tail includes large advance issues. Zeros reflect verified coverage.
Annual issue timing, a customer's prior evidence and the exact horizon can all
affect the distribution. These comparisons cannot isolate a causal effect.

In [ ]:
display(data["Actual GBP"].describe(percentiles=[0.1, 0.5, 0.9, 0.95, 0.99]).to_frame())
display(
    pd.Series(
        {"Zero observations": data["Actual GBP"].eq(0).sum(), "Skewness": data["Actual GBP"].skew()}
    )
)
display(data.groupby("Cadence")["Actual GBP"].agg(["size", "mean", "median", "max"]).round(2))
fig, ax = plt.subplots(figsize=(7.5, 3.5))
ax.hist(data["Actual GBP"], bins=22, color=blue)
ax.set(
    title="Mature six-month value in validation", xlabel="Actual issued GBP", ylabel="Observations"
)
currency(ax.xaxis)
finish()

fig, ax = plt.subplots(figsize=(8, 3.4))
classes = data["Actual GBP"].gt(0).map({False: "Confirmed zero", True: "Positive"})
counts = classes.value_counts()
ax.barh(counts.index, counts.values, color=[blue, grey])
ax.bar_label(ax.containers[0], padding=4)
ax.set(
    title="Mature validation target support",
    xlabel="Customer observations",
    xlim=(0, len(data) * 1.12),
)
finish()

### Shape and monetary ties

Near-zero sample skewness can coexist with a discrete distribution and a material upper tail. Exact amounts recur under the synthetic pricing and billing schedules. The ECDF makes repeated amounts visible without dropping outliers.

In the reference, validation skewness is 0.0506 and 248 of 355 outcomes equal
£474.00. A small skewness statistic does not imply a continuous or normal
distribution.

In [ ]:
ordered = data["Actual GBP"].sort_values(ignore_index=True)
display(data["Actual GBP"].value_counts().head(10).rename("Observations").to_frame())
fig, ax = plt.subplots(figsize=(7.8, 3.4))
ax.step(ordered, (ordered.index + 1) / len(ordered), where="post", color=blue)
ax.set(
    title="Empirical distribution of realised validation value",
    xlabel="Realised issued GBP",
    ylabel="Share of observations at or below",
)
currency(ax.xaxis)
finish()

## 08. Feature engineering

The separate CLV registry records observed account tenure, known lifecycle counts,
visible issuance and cadence, active snapshot amounts, coverage and UTC month.
Usage features are deferred pending a defensible customer-level aggregation.
The 25 episode-level Phase 05 features and Phase 06 churn probabilities are
outside this feature set.

## 09. Feature quality

Missingness reflects evidence limits as of the observation. A missing snapshot
on an active episode suppresses the account's combined snapshot amount. A
constant training feature has no learned variation. Category handling covers
levels absent in training.

In [ ]:
display(
    pd.DataFrame(design["features"])
    .set_index("name")
    .join(pd.DataFrame(report["feature_quality"]).T)
)

### Individual feature dictionary and quality

The registry supplies each definition, source, unit and nullability rule. The
second table adds calculations, interpretation and measured training cardinality.
All evidence is known by observation t. Lifecycle requires occurrence, effective
and availability times. Issues need both issue and matching evidence timestamps.
A snapshot must have an available observation. Missing coverage is distinct
from a true zero on a present ledger. Constant features remain in the frozen
contract, and zero observed missingness does not establish production completeness.

In [ ]:
if "partition_sha256" in design:
    from nowa_customer_intelligence.models.clv.features import FEATURES
    from nowa_customer_intelligence.models.clv.research import model_inputs

    training = partitions["training"]
    assert _fingerprint(training) == design["partition_sha256"]["training"]
    train_features = model_inputs(training)
    val_features = model_inputs(validation)
    dictionary = pd.DataFrame(design["features"]).set_index("name")
    assert tuple(dictionary.index) == tuple(feature.name for feature in FEATURES)
    notes = {
        "tenure_days": (
            "t date minus account creation date",
            "Account age, not continuous service",
        ),
        "episodes": ("Count known canonical establishments", "Lifecycle evidence only"),
        "completed": ("Count known terminations", "Later terminations unavailable"),
        "reactivated": ("Count known reactivations", "Structured fictional transitions"),
        "active": ("Count activated episodes without known termination", "As-known activity"),
        "invoice_count": ("Count unique visible issues", "Null without a feed"),
        "issued_gbp": ("Sum visible issue amounts", "Window-bounded; null without a feed"),
        "recent_gbp": (
            "Sum on the last 90 UTC dates",
            "May use a shorter source history; null without a feed",
        ),
        "mean_invoice_gbp": (
            "Visible sum divided by invoice count",
            "Null without a visible issue",
        ),
        "recency_days": ("t date minus latest visible issue date", "Null without a visible issue"),
        "monthly": ("Count visible monthly bills", "Null without a feed"),
        "annual": ("Count visible annual bills", "Null without a feed"),
        "billing_evidence": ("Unavailable / no issue / issued evidence", "Coverage category"),
        "cadence": (
            "Monthly / annual / mixed / unobserved from visible counts",
            "May change across dates",
        ),
        "snapshot_gbp": (
            "Sum usable active episode rates if all covered",
            "Null if incomplete; zero if none",
        ),
        "snapshot_coverage": ("Count active episodes with usable rates", "Coverage count"),
        "month": ("UTC two-digit month", "Calendar proxy rather than a contract property"),
    }
    dictionary["Type"] = (
        dictionary["units"]
        .map({"GBP": "Decimal", "GBP/month": "Decimal", "category": "String"})
        .fillna("Integer")
    )
    dictionary["Available"] = dictionary["source"].map(
        {
            "Customer": "Creation by t",
            "Lifecycle": "Lifecycle evidence by t",
            "Invoice": "Issue and evidence by t",
            "Snapshot": "Usable snapshot by t",
            "Calendar": "t",
        }
    )
    dictionary["Calculation"] = [notes[key][0] for key in dictionary.index]
    dictionary["Limits and missingness"] = [notes[key][1] for key in dictionary.index]
    quality = pd.DataFrame(report["feature_quality"]).T
    assert quality["training_distinct"].to_dict() == train_features.nunique().to_dict()
    assert quality["training_missing"].to_dict() == train_features.isna().sum().to_dict()
    assert quality["validation_missing"].to_dict() == val_features.isna().sum().to_dict()
    display(dictionary[["definition", "source", "units", "Type", "nullable", "Available"]])
    display(dictionary[["Calculation", "Limits and missingness"]].join(quality))
    distinct = quality["training_distinct"].sort_values()
    constants = distinct.index[distinct.eq(1)].tolist()
    display(Markdown(f"**Constant in training** {', '.join(constants)}."))
    fig, ax = plt.subplots(figsize=(8.5, 4.7))
    ax.barh(distinct.index, distinct, color=[grey if v == 1 else blue for v in distinct])
    ax.set_xscale("log")
    ax.set(title="Observed raw feature cardinality", xlabel="Distinct training values · log scale")
    finish()

### Development predictor relationships

Compare raw distributions before scaling. Rank correlations omit constants,
which have undefined correlations. The actual-value scatter is a post-hoc
development diagnostic and was outside the frozen selection. Cadence and
observation month distributions may move with chronology and enrolment.

In [ ]:
if "partition_sha256" in design:
    import numpy as np

    fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.4))
    for ax, names in zip(axes, ("issued_gbp", "tenure_days", "recency_days"), strict=True):
        both = pd.concat([train_features[names], val_features[names]]).dropna()
        edges = np.linspace(both.min(), both.max(), 15)
        for sample, label, colour in (
            (train_features, "Training", blue),
            (val_features, "Validation", green),
        ):
            values = sample[names].dropna()
            ax.hist(
                values,
                bins=edges,
                weights=np.full(len(values), 1 / len(values)),
                label=label,
                histtype="step",
                lw=2,
                color=colour,
            )
        ax.set(
            title=names.replace("_", " ").title(),
            ylabel="Share of visible rows",
            xlabel="GBP" if names == "issued_gbp" else "Days",
        )
        if names == "issued_gbp":
            currency(ax.xaxis)
    axes[0].legend()
    finish()
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.5))
    for ax, field in zip(axes, ("cadence", "month"), strict=True):
        shares = (
            pd.DataFrame(
                {
                    "Training": train_features[field].value_counts(normalize=True),
                    "Validation": val_features[field].value_counts(normalize=True),
                }
            )
            .fillna(0)
            .sort_index()
        )
        shares.plot.bar(ax=ax, rot=0, color=[blue, green])
        ax.set(title=f"Observed {field} profile", xlabel=field.title(), ylabel="Share of rows")
    finish()
    numeric = train_features.select_dtypes("number")
    corr = numeric.loc[:, numeric.nunique().gt(1)].corr(method="spearman")
    fig, ax = plt.subplots(figsize=(8, 5.1))
    matrix = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set(
        xticks=range(len(corr)),
        xticklabels=corr.columns,
        yticks=range(len(corr)),
        yticklabels=corr.index,
        title="Training financial predictor rank correlations",
    )
    ax.tick_params(axis="x", labelrotation=55)
    ax.grid(False)
    fig.colorbar(matrix, ax=ax, label="Spearman correlation")
    finish()
    fig, axes = plt.subplots(1, 3, figsize=(11.6, 3.4))
    diagnostic = val_features.assign(**{"Actual GBP": data["Actual GBP"]})
    for ax, left, right in zip(
        axes,
        ("invoice_count", "tenure_days", "issued_gbp"),
        ("issued_gbp", "issued_gbp", "Actual GBP"),
        strict=True,
    ):
        ax.scatter(diagnostic[left], diagnostic[right], alpha=0.4, s=13, color=blue)
        ax.set(
            title="Post-hoc target comparison"
            if right == "Actual GBP"
            else "Historical relationship",
            xlabel=left.replace("_", " ").capitalize(),
            ylabel=right.replace("_", " ").capitalize(),
        )
        currency(ax.yaxis)
        if left == "issued_gbp":
            currency(ax.xaxis)
    finish()
    fig, ax = plt.subplots(figsize=(7.5, 3.4))
    val_features.boxplot(column="recency_days", by="cadence", ax=ax)
    fig.suptitle("")
    ax.set(
        title="Validation invoice recency by cadence",
        xlabel="Historical cadence",
        ylabel="Days since latest visible issue",
    )
    finish()

## 10. Temporal validation

Seed 42 training observations run June 2021 to June 2022. Validation uses January
and February 2023. Each outcome must mature before the next exclusive boundary.
The independent seed-731 reserve uses September 2023 month end. Its horizon ends
in March 2024. Calendar overlap in past lookbacks is possible across disjoint
accounts and populations.

## 11. Leakage prevention

Historical rows come from a UTC as-of view. Target values come from a separate
future invoice index. Later reactivations contribute only to eligible outcomes.
The partition fingerprint and fitted artefact were recorded before reserve inference.

In [ ]:
s = pd.DataFrame(design["support"]).T

### Maturity alongside observation dates

The second panel shows the six-month horizon ends. Development evidence must mature before the next exclusive boundary, independently of the account allocation. The reserve uses the separate source extraction cutoff.

In [ ]:
if "windows" in design:
    from nowa_customer_intelligence.models.clv.target import six_month_end

    bounds = pd.to_datetime(design["windows"], utc=True, format="ISO8601")
    fig, axes = plt.subplots(2, 1, figsize=(9.5, 4.1), sharex=True)
    for idx, (_, row) in enumerate(s.iterrows()):
        anchors = pd.to_datetime([row["first_observation"], row["last_observation"]], utc=True)
        axes[0].plot(anchors, [idx, idx], "o-", color=(blue, green, grey)[idx], lw=4)
        ends = [six_month_end(value.to_pydatetime()) for value in anchors]
        axes[1].plot(ends, [idx, idx], "o-", color=(blue, green, grey)[idx], lw=4)
    for ax, title in zip(axes, ("Observation span", "Six-month end span"), strict=True):
        for index in (2, 4, 5):
            ax.axvline(bounds[index], ls=":", color="#364556")
        ax.set(title=title, yticks=range(len(s)), yticklabels=s.index)
    axes[1].set(xlabel="UTC date · dotted lines mark maturity cutoffs")
    finish()

## 12. Preprocessing pipelines

Numeric imputation and scaling and category encoding are learned exclusively
on training data. Nullable evidence carries indicators where observed in
training. Unknown categories transform without refitting. Decimal amounts are
converted at the numerical model boundary. Predictions are clipped at zero
under the predeclared non-negative monetary constraint.

## 13. Statistical baselines

The median of training outcomes provides the absolute-error benchmark. A
contractual schedule baseline remains deferred because future retention and
renewal rules lack evidence. The observed snapshot rate is reported separately
in notebook 07.

## 14. Linear regression

Ridge shrinkage reduces the influence of unstable correlated features.

## 15. Random Forest regression

Leaf minimums and a depth cap limit sensitivity to small customer cohorts.

## 16. Gradient Boosting regression

Histogram boosting uses shallow controlled leaves and disables internal
early stopping to keep the fixed validation protocol.

## 17. Model configuration table

Training uses the fixed menu shown below. Model seed 42 governs stochastic
candidates and uncertainty resampling.

### Preprocessing architecture and model mathematics

All four saved estimators use the same column transformer. The numeric path
uses medians, adds missingness indicators only for columns missing in training
and standardises values. Unknown categories map to an all-zero one-hot pattern.
Scaling matters for the Ridge penalty, whereas tree splits principally depend
on ordering. All transforms are trained on training rows, and constants are
retained. There is no feature selector. Outputs are clipped to non-negative GBP.

The median benchmark solves (argmin_m sum_i|y_i-m|). As (m) rises, absolute
loss grows for rows on the left and falls for rows on the right. A median balances
those counts. A flat forecast cannot represent non-uniform billing timing.

Ridge uses (hat y=eta_0+x^mathsf{T}eta) and minimises
(sum_i(y_i-hat y_i)^2+alpha|eta|^2_2). The saved (alpha=10) penalises
large coefficients. Scaling makes the relative penalty meaningful across units.
Shrinkage can stabilise correlated effects at the cost of bias. Without encoded
interactions the model remains additive and can miss charging thresholds.

A squared-loss regression tree chooses splits that reduce within-node squared
error and predicts the mean in each leaf. The Random Forest averages
(hat y(x)=B^{-1}sum_b T_b(x)) across (B=100) bootstrap trees. Random feature
subsampling is a general forest mechanism. This saved model uses all transformed
features at each split because the fraction is 1.0. Depth eight and a minimum of
15 rows per leaf limit small local fits. Bagging can reduce variance but cannot
resolve calendar shift or reliably extrapolate beyond the training outcomes.

Histogram boosting uses (F_m=F_{m-1}+eta f_m(x)). Under squared error, the
negative gradient is proportional to (y-F_{m-1}(x)). Numeric histograms
approximate split candidates. The saved update size 0.06, 100 rounds, 15 leaves,
minimum 25 rows per leaf and L2 leaf penalty 1 constrain flexibility. Internal
early stopping is disabled under the locked temporal protocol. The model can
represent nonlinear interactions and may also learn scenario-specific value atoms.

In [ ]:
display(
    pd.DataFrame(
        [
            (
                name,
                "Train-only median, conditional indicators and scaling",
                "Train-only unknown-safe one-hot",
                "No selector; non-negative GBP clipping",
            )
            for name in design["configurations"]
        ],
        columns=["Estimator", "Numeric path", "Categorical path", "Output"],
    )
)
fig, ax = plt.subplots(figsize=(11, 3))
ax.axis("off")
ax.set(xlim=(0, 1), ylim=(0, 1), title="Fitted preprocessing and forecast path")
items = [
    ("CLV registry\nDecimal GBP", 0.09, 0.58),
    ("Numeric model\nboundary", 0.28, 0.58),
    ("Training median and\nscaling", 0.50, 0.80),
    ("Training category\nencoding", 0.50, 0.34),
    ("Frozen estimator", 0.73, 0.58),
    ("Floor at zero GBP", 0.92, 0.58),
]
for label, x, y in items:
    ax.text(
        x,
        y,
        label,
        ha="center",
        va="center",
        bbox={"facecolor": "#edf4fa", "edgecolor": blue, "boxstyle": "round,pad=.36"},
    )
for start, end in ((0, 1), (1, 2), (1, 3), (2, 4), (3, 4), (4, 5)):
    ax.annotate(
        "",
        xy=items[end][1:],
        xytext=items[start][1:],
        arrowprops={"arrowstyle": "->", "color": grey, "shrinkA": 43, "shrinkB": 43},
    )
finish()

In [ ]:
if "partition_sha256" in design:
    from hashlib import sha256
    from importlib.metadata import version
    from io import BytesIO

    from joblib import load

    model_file = workspace / "results" / "selected.joblib"
    trusted_digest = "250f4d860a662f69f95e2577cd890dc4f86d75c33771bc8dc0ed04aa4398afc7"
    if model_file.is_file() and version("scikit-learn") == design["dependencies"]["scikit-learn"]:
        blob = model_file.read_bytes()
        assert sha256(blob).hexdigest() == report["frozen"]["model_sha256"] == trusted_digest
        frozen = load(BytesIO(blob))
        transform, estimator = [step[1] for step in frozen.steps]
        names = transform.get_feature_names_out()
        assert len(names) == estimator.n_features_in_
        display(
            pd.Series(
                {
                    "Raw inputs": transform.n_features_in_,
                    "Transformed columns": len(names),
                    "Missingness indicators": sum("missingindicator" in name for name in names),
                },
                name="Verified fitted dimensions",
            ).to_frame()
        )
        encoder = transform.named_transformers_["category"]
        display(
            pd.DataFrame(
                [
                    (name, len(levels), ", ".join(map(str, levels)))
                    for name, levels in zip(
                        encoder.feature_names_in_, encoder.categories_, strict=True
                    )
                ],
                columns=["Encoded feature", "Columns", "Observed training levels"],
            )
        )
    else:
        display(
            Markdown("Fitted dimensions require the retained model and matching library version.")
        )

### Training-median benchmark reconciliation

The saved observation-level forecasts are for the winner. Recover the baseline constant from the fingerprinted training targets and validate its error against the archived benchmark metrics.

In the reference, the constant is £474.00 and 248 of 355 rows are exact matches.
Among the other 107 rows, absolute error averages £374.77. This explains why
zero median AE and £112.96 MAE can both be correct.

In [ ]:
if "partition_sha256" in design:
    from decimal import Decimal
    from statistics import median

    assert design["configurations"]["Training median"]["strategy"] == "median"
    constant = median(r.target.future_6m_issued_gbp for r in training)
    outcomes = [Decimal(str(value)) for value in data["Actual GBP"]]
    errors_from_constant = [abs(value - constant) for value in outcomes]
    exact = sum(value == 0 for value in errors_from_constant)
    assert abs(float(sum(errors_from_constant) / len(outcomes)) - base_mae) < 1e-9
    assert float(median(errors_from_constant)) == scores.at["Training median", "Median AE"]
    below = sum(v < constant for v in outcomes)
    above = sum(v > constant for v in outcomes)
    elsewhere = (
        sum(errors_from_constant) / (len(outcomes) - exact) if exact < len(outcomes) else None
    )
    display(
        pd.Series(
            {
                "Training constant": f"£{constant:,.2f}",
                "Exact matches": f"{exact} / {len(outcomes)} ({exact / len(outcomes):.2%})",
                "Below / above the constant": f"{below} / {above}",
                "MAE among non-matches": f"£{elsewhere:,.2f}"
                if elsewhere is not None
                else "Undefined",
                "Largest error": f"£{max(errors_from_constant):,.2f}",
            },
            name="Verified median baseline",
        ).to_frame()
    )

In [ ]:
meaning = {
    "strategy": "Training-only constant statistic",
    "alpha": "L2 penalty strength",
    "fit_intercept": "Fit an unpenalised intercept",
    "solver": "Library chooses a numerical solver",
    "n_estimators": "Number of bootstrap trees",
    "criterion": "Tree split loss",
    "bootstrap": "Resample training rows with replacement",
    "max_samples": "None draws the training row count per tree",
    "max_depth": "Depth cap or no cap when None",
    "min_samples_leaf": "Minimum number of rows per leaf",
    "max_features": "Fraction of transformed features eligible for a split",
    "max_iter": "Solver iteration limit or boosting rounds",
    "max_leaf_nodes": "Maximum leaves per tree",
    "learning_rate": "Shrinkage per update",
    "l2_regularization": "Penalty on boosting leaf values",
    "early_stopping": "Whether internal stopping is enabled",
    "max_bins": "Maximum numeric histogram bins per feature",
    "loss": "Boosting objective",
    "random_state": "Seed where the estimator uses one",
    "n_jobs": "Worker limit",
    "validation_fraction": "Inactive when early stopping is disabled",
    "categorical_features": "Automatic dtype detection after numerical encoding",
}
for name, parameters in design["configurations"].items():
    display(Markdown(f"**{name} · saved parameters**"))
    display(
        pd.DataFrame(
            [
                (
                    key,
                    repr(value),
                    meaning.get(key, "Saved setting. No separate rationale recorded."),
                )
                for key, value in parameters.items()
            ],
            columns=["Parameter", "Value", "Technical interpretation"],
        )
    )

## 18. Evaluation metrics

MAE in GBP is the primary comparison. RMSE emphasises large errors. Median
absolute error describes a typical observation. Positive signed bias means
overforecasting. Aggregate error sums signed errors across observations, so
repeat observations should not be interpreted as one portfolio forecast.

## 19. Model comparison dashboard

All candidates use the same validation population. Compare against the
training median before interpreting small improvements.

The absolute reduction subtracts candidate validation MAE from the median's
validation MAE. Divide by the median's MAE for a relative reduction. Boosting
led on MAE, while the forest had lower RMSE. A small difference in typical
error can coexist with a larger penalty on extreme errors.

### Metric definitions and worked arithmetic

For (e_i=hat y_i-y_i), MAE is (sum_i|e_i|/n), RMSE is
(sqrt{sum_i e_i^2/n}), and median AE is (mathrm{median}(|e_i|)).
Bias is (sum_i e_i/n) and aggregate error is (sum_i e_i). Each average
is GBP per observation. The aggregate is GBP summed over the evaluation rows.
(R^2=1-sum_i e_i^2/sum_i(y_i-ar y)^2) when outcomes vary.
Spearman is the Pearson correlation between average-tie ranks. Both are
dimensionless diagnostics. Only validation MAE governs selection.

For illustration, realised values £100, £200, £0 and forecasts £110, £160,
£30 have errors +£10, −£40, +£30. MAE is £26.67, RMSE £29.44 and median AE
£30. Bias and aggregate error are zero, even though individual misses remain.
(R^2=0.87) and Spearman is 1.00. Correct rank order can coexist with poor
monetary accuracy.

In [ ]:
table = scores.copy()
table["Training MAE"] = pd.Series(
    {name: row["training"]["MAE"] for name, row in report["scores"].items()}
)
median_mae = scores.at["Training median", "MAE"]
table["MAE reduction GBP"] = median_mae - table["MAE"]
table["MAE reduction (%)"] = 100 * table["MAE reduction GBP"] / median_mae if median_mae else pd.NA
display(table.round(2))
fig, ax = plt.subplots(figsize=(8, 3.6))
scores[["MAE", "RMSE"]].plot.barh(ax=ax, color=[blue, green])
ax.set(title="Validation financial error by candidate", xlabel="GBP per observation", ylabel="")
currency(ax.xaxis)
finish()

## 20. Predictions versus actuals

The diagonal marks equal forecast and actual. Clustering near one forecast
would limit discrimination between higher and lower issued amounts.

## 21. Residual analysis

Use signed errors to see whether large errors occur on one or both sides of
zero. Zero-value customers can contribute large overpredictions.

## 22. Underprediction and overprediction

Underprediction understates expected billed exposure. Overprediction can
inflate planning assumptions. Neither error is an estimate of lost profit.

The reference validation median absolute error was £10.03 compared with a
mean absolute error of £58.27 and an RMSE of £152.26. A relatively small
typical error coexists with a much wider tail, so budgeting on a single
error statistic would miss the larger forecast misses.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.hist(data["Error GBP"], bins=22, color=green)
ax.axvline(0, ls="--", color=grey)
ax.set(title="Validation signed errors", xlabel="Prediction minus actual, GBP", ylabel="Rows")
currency(ax.xaxis)
finish()
display(data["Error GBP"].agg(["min", "median", "mean", "max"]).to_frame())

fig, ax = plt.subplots(figsize=(7, 3.3))
ax.hist(data["Absolute error GBP"], bins=20, color=blue)
ax.set(
    title="Validation absolute forecast errors", xlabel="Absolute error, GBP", ylabel="Observations"
)
currency(ax.xaxis)
finish()
display(
    data[["Error GBP", "Absolute error GBP"]].quantile([0, 0.25, 0.5, 0.75, 0.9, 0.99, 1]).round(2)
)
direction = data["Error GBP"].map(
    lambda val: "Overforecast" if val > 0 else "Underforecast" if val < 0 else "Exact"
)
display(data.groupby(direction)["Error GBP"].agg(["size", "mean", "median", "sum"]).round(2))
actual, pred = data["Actual GBP"], data["Predicted GBP"]
assert (
    abs(scores.at[selected, "MAE"] - data["Absolute error GBP"].mean()) < 1e-9
    or "partition_sha256" not in design
)
if "partition_sha256" in design:
    assert abs(scores.at[selected, "RMSE"] - ((actual - pred).pow(2).mean() ** 0.5)) < 1e-9
    assert abs(scores.at[selected, "Bias"] - data["Error GBP"].mean()) < 1e-9
    assert abs(scores.at[selected, "Aggregate error"] - data["Error GBP"].sum()) < 1e-7

### Agreement, tail risk and directional reconciliation

The equal-scale comparison highlights how similar realised amounts can receive
different forecasts by cadence. The forecast distribution shows concentration
of predicted amounts. Residual plots retain the sign convention and show both
axes against the two financial quantities. The ECDF reports absolute-error
coverage without imposing a normal distribution.

In [ ]:
limit = data[["Actual GBP", "Predicted GBP"]].max().max() * 1.04
fig, ax = plt.subplots(figsize=(5.4, 4.6))
for cadence, group in data.groupby("Cadence"):
    ax.scatter(group["Actual GBP"], group["Predicted GBP"], label=cadence, s=16, alpha=0.52)
ax.plot([0, limit], [0, limit], "--", color=grey)
ax.set(
    xlim=(0, limit),
    ylim=(0, limit),
    aspect="equal",
    title="Equal-scale validation agreement by cadence",
    xlabel="Realised GBP",
    ylabel="Forecast GBP",
)
currency(ax.xaxis)
currency(ax.yaxis)
ax.legend()
finish()
fig, ax = plt.subplots(figsize=(7.8, 3.4))
for field, style in (("Actual GBP", "-"), ("Predicted GBP", "--")):
    values = data[field].sort_values(ignore_index=True)
    ax.step(values, (values.index + 1) / len(values), where="post", ls=style, label=field)
ax.set(
    title="Forecast and outcome distributions in validation",
    xlabel="GBP per observation",
    ylabel="Share of rows at or below",
)
currency(ax.xaxis)
ax.legend()
finish()
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, field in zip(axes, ("Predicted GBP", "Actual GBP"), strict=True):
    ax.scatter(data[field], data["Error GBP"], s=12, alpha=0.4, color=blue)
    ax.axhline(0, ls="--", color=grey)
    ax.set(
        title=f"Signed error versus {field.lower()}",
        xlabel=field,
        ylabel="Forecast minus realised GBP",
    )
    currency(ax.xaxis)
    currency(ax.yaxis)
finish()
errors = data["Absolute error GBP"].sort_values(ignore_index=True)
fig, ax = plt.subplots(figsize=(7.8, 3.4))
ax.step(errors, (errors.index + 1) / len(errors), where="post", color=blue)
ax.set(
    title="Validation absolute-error coverage",
    xlabel="Absolute error, GBP",
    ylabel="Share of observations within error",
)
currency(ax.xaxis)
finish()

In [ ]:
direction = data["Error GBP"].map(
    lambda value: "Overforecast" if value > 0 else "Underforecast" if value < 0 else "Exact"
)
signs = data.groupby(direction)["Error GBP"].agg(["size", "mean", "median", "sum"])
signs["Observation share"] = signs["size"].map(lambda size: f"{size / len(data):.2%}")
display(signs.round(2))
over, under = signs["sum"].get("Overforecast", 0), signs["sum"].get("Underforecast", 0)
net = over + under
if "partition_sha256" in design:
    assert abs(net - scores.at[selected, "Aggregate error"]) < 1e-7
fig, ax = plt.subplots(figsize=(7.4, 3.3))
ax.bar(
    ["Overforecast", "Underforecast", "Net"],
    [over, under, net],
    bottom=[0, over, 0],
    color=[blue, "#a4513c", grey],
)
ax.axhline(0, color="#354558", lw=0.8)
ax.set(
    title="Directional reconciliation of validation error",
    ylabel="Sum of signed GBP across evaluated observations",
)
currency(ax.yaxis)
finish()

### Anonymised large misses and confirmed zeros

Local references identify rows within the saved validation file. The extreme
examples remain in evaluation. A confirmed zero covers a complete, observed
window with no issuance. It can occur during annual service and cannot establish
churn or collection. These descriptive patterns do not determine causes.

Seven of the ten largest absolute errors are underforecasts. Six of those ten
rows have observed annual cadence and four have monthly cadence. The ten rows
account for 35.79% of absolute error and 67.62% of squared error. The 42
confirmed-zero rows, 38 annual and four mixed, contribute 23.53% of total
absolute error. These are overlapping descriptive summaries.

In [ ]:
largest = data.nlargest(min(10, len(data)), "Absolute error GBP").copy()
largest.index = [f"V{index + 1:03d}" for index in largest.index]
display(largest[["Observation month", "Cadence", "Tenure days", "Historical GBP"]].round(2))
display(largest[["Predicted GBP", "Actual GBP", "Error GBP", "Absolute error GBP"]].round(2))
l1 = data["Absolute error GBP"].sum()
l2 = data["Error GBP"].pow(2).sum()
if l1 and l2:
    display(
        pd.Series(
            {
                "Top ten share of absolute error": largest["Absolute error GBP"].sum() / l1,
                "Top ten share of squared error": largest["Error GBP"].pow(2).sum() / l2,
            }
        )
        .map("{:.2%}".format)
        .to_frame("Error concentration")
    )
zero = data.loc[data["Actual GBP"].eq(0)]
if not zero.empty:
    display(
        zero.groupby("Cadence")
        .agg(
            Observations=("Actual GBP", "size"),
            Mean_forecast_GBP=("Predicted GBP", "mean"),
            Median_forecast_GBP=("Predicted GBP", "median"),
            Maximum_forecast_GBP=("Predicted GBP", "max"),
            Absolute_error_GBP=("Absolute error GBP", "sum"),
        )
        .round(2)
    )
    if l1:
        display(
            Markdown(
                f"Confirmed zeros contributed **{zero['Absolute error GBP'].sum() / l1:.2%}** "
                "of validation absolute error."
            )
        )
    fig, ax = plt.subplots(figsize=(7.5, 3.2))
    ax.hist(zero["Predicted GBP"], bins=18, color=green)
    ax.set(
        title="Forecasts for confirmed-zero validation targets",
        xlabel="Forecast issued GBP",
        ylabel="Validation observations",
    )
    currency(ax.xaxis)
    finish()

## 23. Error by customer-value band

Historical bands use observed issued GBP at prediction time. Realised bands
use targets retrospectively. Fixed illustrative boundaries avoid fitting
band cut-offs to the reserve.

## 24. Error by billing frequency

Historical cadence and annual prepayment can materially shift errors.

## 25. Error by observation cohort

Compare month and cadence group support with the errors. Small groups give
weak evidence, and rows that share an account are dependent.

The reference monthly, annual and mixed groups contain 266, 81 and eight rows
respectively. Distinct accounts are shown separately, and can appear in different
cadence groups at different dates. The mixed group's error is especially unstable.
Annual prepayment, feature coverage and which invoices fall inside the horizon
are plausible contributors to the cadence contrast.

In the reference, monthly MAE was £32.13 and annual MAE was £120.72.
Median absolute errors were £6.02 and £47.63 respectively, so the contrast
was visible beyond mean error. The mixed group had £294.95 MAE but only
four distinct accounts. The lowest realised band was overforecast by £136.95
on average and the highest was underforecast by £113.93. Opposite errors can
partly cancel in the aggregate. These retrospective bands explain observed
error patterns and cannot identify accounts at forecast time.

In [ ]:
data["Historical band"] = pd.cut(data["Historical GBP"], [-1, 500, 1500, float("inf")])
data["Realised band"] = pd.cut(data["Actual GBP"], [-1, 250, 750, float("inf")])


def cohort(field):
    return (
        data.groupby(field, observed=True)
        .agg(
            Observations=("Actual GBP", "size"),
            Accounts=("Account key", "nunique"),
            Mean_target=("Actual GBP", "mean"),
            Median_target=("Actual GBP", "median"),
            MAE=("Absolute error GBP", "mean"),
            Median_absolute_error=("Absolute error GBP", "median"),
            Mean_signed_error=("Error GBP", "mean"),
            Median_signed_error=("Error GBP", "median"),
        )
        .assign(Accounts=lambda rows: rows["Accounts"].where(rows["Accounts"].gt(0)))
        .round(2)
    )


for field in ("Historical band", "Realised band", "Cadence", "Observation month"):
    display(cohort(field))
assert cohort("Cadence")["Observations"].sum() == len(data)
for field, title, order in (
    ("Cadence", "Billing cadence", sorted(data["Cadence"].unique())),
    ("Realised band", "Realised six-month value", list(data["Realised band"].cat.categories)),
):
    observed = [key for key in order if data[field].eq(key).any()]
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
    for ax, measure in zip(axes, ("Actual GBP", "Absolute error GBP"), strict=True):
        samples = [data.loc[data[field].eq(key), measure] for key in observed]
        labels = [f"{key} (n={len(sample)})" for key, sample in zip(observed, samples, strict=True)]
        ax.boxplot(samples, tick_labels=labels)
        ax.set(title=measure, xlabel=title, ylabel="GBP per validation observation")
        ax.tick_params(axis="x", labelrotation=14)
        currency(ax.yaxis)
    finish()

### Tenure, cadence contribution and retrospective value bands

Tenure bands use fixed boundaries at 180 and 365 days. A historical group may
be available at prediction time, while a realised-value band is only known
after outcome maturity. Cadence contributions sum absolute errors across
observations and require support as well as per-row error statistics.

In [ ]:
data["Tenure band"] = pd.cut(
    data["Tenure days"],
    [-1, 179, 364, float("inf")],
    labels=["Under 180 days", "180–364 days", "365 days or more"],
)
display(cohort("Tenure band"))
cadence_sum = data.groupby("Cadence").agg(
    Observations=("Actual GBP", "size"),
    Forecast_mean=("Predicted GBP", "mean"),
    Forecast_median=("Predicted GBP", "median"),
    Absolute_error_GBP=("Absolute error GBP", "sum"),
)
if l1:
    cadence_sum["Share of absolute error"] = cadence_sum["Absolute_error_GBP"] / l1
display(cadence_sum.round(4))
box_groups = list(data.groupby("Realised band", observed=True)["Predicted GBP"])
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.boxplot([values for _, values in box_groups], tick_labels=[str(key) for key, _ in box_groups])
ax.set(
    title="Forecast distribution within retrospective realised-value bands",
    xlabel="Realised GBP band",
    ylabel="Forecast GBP",
)
currency(ax.yaxis)
finish()

## 26. Training versus validation

Training errors are in-sample. Gaps may reflect overfitting, differences in
cohort and chronology, or both.

## 27. Customer value ranking

Spearman correlation describes association between ranks in validation.
Ties and repeated observations limit interpretation. Ranking is secondary
to the approved GBP error criterion.

## 28. Financial aggregate accuracy

Compare the sum of observation-level forecasts with the sum of realised
issues on the exact same validation rows. Repeated windows can count shared
future charges multiple times, so these totals are evaluation diagnostics.

In [ ]:
comparison = pd.DataFrame(
    {
        name: {part: m["MAE"] for part, m in metrics.items()}
        for name, metrics in report["scores"].items()
    }
).T
display(comparison.round(2))
ax = comparison.plot.barh(figsize=(8, 3.6), color=[blue, green])
ax.set(title="In-sample and forward validation MAE", xlabel="GBP per observation", ylabel="")
currency(ax.xaxis)
finish()
display(
    pd.Series(
        {
            "Validation Spearman correlation": data[["Actual GBP", "Predicted GBP"]]
            .corr(method="spearman")
            .iloc[0, 1],
            "Summed actual GBP": data["Actual GBP"].sum(),
            "Summed predicted GBP": data["Predicted GBP"].sum(),
            "Aggregate signed error GBP": data["Error GBP"].sum(),
        }
    )
    .round(2)
    .to_frame()
)

### Post-hoc ranking and financial capture

Each distinct forecast gets one rank group. Identical forecasts enter the
capture curve together. For a top-decile overlap, include all ties at each
boundary and report the resulting row counts. This is a post-hoc development
description, with repeat observations in the denominator. It is outside model
selection and does not define commercial segments.

The tie-inclusive groups contain 36 forecast-ranked rows and 42 realised-ranked
rows with 34 in common. The forecast group captures 18.09% of realised GBP
across 355 validation observations. The result is conditional on this row
population and the boundary tie convention.

In [ ]:
ranked = (
    data.groupby("Predicted GBP")
    .agg(
        Rows=("Actual GBP", "size"),
        Realised=("Actual GBP", "sum"),
        Mean_realised=("Actual GBP", "mean"),
    )
    .sort_index(ascending=False)
)
ranked["Cumulative row share"] = ranked["Rows"].cumsum() / len(data)
total_realised = ranked["Realised"].sum()
if total_realised:
    ranked["Cumulative GBP share"] = ranked["Realised"].cumsum() / total_realised
    fig, ax = plt.subplots(figsize=(7.8, 3.5))
    ax.plot(
        [0, *ranked["Cumulative row share"]],
        [0, *ranked["Cumulative GBP share"]],
        color=blue,
        label="Descending forecast groups",
    )
    ax.plot([0, 1], [0, 1], "--", color=grey, label="Equal-share reference")
    ax.set(
        title="Post-hoc capture of realised validation issuance",
        xlabel="Share of validation observations",
        ylabel="Share of realised GBP",
    )
    ax.legend()
    finish()
    amount = (len(data) + 9) // 10
    top_forecast = data["Predicted GBP"].ge(data["Predicted GBP"].nlargest(amount).iloc[-1])
    top_actual = data["Actual GBP"].ge(data["Actual GBP"].nlargest(amount).iloc[-1])
    capture = data.loc[top_forecast, "Actual GBP"].sum() / total_realised
    display(
        pd.Series(
            {
                "Top forecast rows including cutoff ties": int(top_forecast.sum()),
                "Top actual rows including cutoff ties": int(top_actual.sum()),
                "Overlap rows": int((top_forecast & top_actual).sum()),
                "Realised GBP captured": f"{capture:.2%}",
            },
            name="Top-decile descriptive overlap",
        ).to_frame()
    )
display(ranked.head(8).round(3))
display(
    pd.DataFrame(
        [
            (
                label,
                len(group),
                group[["Actual GBP", "Predicted GBP"]].corr(method="spearman").iloc[0, 1],
            )
            for label, group in data.groupby("Cadence")
        ],
        columns=["Historical cadence", "Observations", "Spearman"],
    )
)

### Generalisation gap and aggregate comparison by observation date

The existing shared-scale plot compares in-sample with later, disjoint-account
MAE. A positive gap is compatible with overfitting or temporal and account
shift. Both can occur. The period table sums evaluated rows. Shared invoices may
fall in overlapping horizons from repeat accounts. Positive and negative errors
partly cancel, so the sum is a diagnostic and cannot be read as a unique portfolio forecast.

In [ ]:
display(
    comparison.assign(
        **{
            "Gap GBP": comparison["validation"] - comparison["training"],
            "Gap relative to training": comparison["validation"]
            / comparison["training"].where(comparison["training"].ne(0))
            - 1,
        }
    ).round(3)
)
monthly_totals = data.groupby("Observation month").agg(
    Actual_GBP=("Actual GBP", "sum"),
    Forecast_GBP=("Predicted GBP", "sum"),
    Error_GBP=("Error GBP", "sum"),
    Observations=("Actual GBP", "size"),
)
monthly_totals["Relative error (%)"] = (
    100
    * monthly_totals["Error_GBP"]
    / monthly_totals["Actual_GBP"].where(monthly_totals["Actual_GBP"].ne(0))
)
display(monthly_totals.round(2))
ax = monthly_totals[["Actual_GBP", "Forecast_GBP"]].plot.bar(
    figsize=(7.8, 3.3), color=[blue, green], rot=0
)
ax.set(
    title="Validation financial totals by observation month",
    xlabel="UTC observation month",
    ylabel="Summed target-window GBP across rows",
)
currency(ax.yaxis)
finish()

## 29. Uncertainty

The finite-sample intervals resample whole accounts. They condition on the fixed
fit and omit model-selection and generator uncertainty.

## 30. Model selection

The frozen rule uses validation MAE and a one-penny tolerance in declared menu
order. The independent reserve had no role in this choice.

## 31. Independent holdout

Only the frozen nominated candidate was scored on the separately generated
reserve. Its later calendar and fresh synthetic seed do not establish real-world
generalisation. There is no cross-model holdout comparison.

The lower reserve MAE comes from one later observation date and a separate
fictional population. The reserve contains 802 distinct accounts, one row each.
No independent baseline comparison exists. The intervals do not capture full
selection uncertainty and cannot establish a significant model difference.

In [ ]:
winner_mae = scores.at[selected, "MAE"]
median_mae = scores.at["Training median", "MAE"]
display(
    Markdown(
        f"**Frozen candidate** {selected}. Validation MAE £{winner_mae:,.2f} "
        f"against £{median_mae:,.2f} for the median baseline."
    )
)
display(pd.DataFrame({"Holdout": report["reserve"]}).round(2))
display(
    pd.DataFrame(
        [report["validation_mae_interval"], report["reserve_mae_interval"]],
        index=["Validation", "Independent reserve"],
        columns=["MAE 2.5%", "MAE 97.5%"],
    ).round(2)
)

### Saved reserve results and conditional uncertainty

The separate seed-731 synthetic reserve has one observation date and one saved
frozen nominee. There is no independent baseline comparator. The charts use
aggregate evidence only. Account resampling keeps repeat observations together.
These intervals are for conditional MAE. They omit full model-selection and
synthetic-generator uncertainty and are not individual prediction intervals.

In [ ]:
scopes = pd.DataFrame(
    {
        "Validation": report["scores"][selected]["validation"],
        "Independent synthetic reserve": report["reserve"],
    }
).T
display(
    scopes.join(
        headline[denominators].rename(
            index={"validation": "Validation", "reserve": "Independent synthetic reserve"}
        )
    ).round(3)
)
ax = scopes[["MAE", "RMSE", "Median AE", "Bias"]].T.plot.barh(figsize=(8, 3.5), color=[blue, green])
ax.set(
    title="Frozen nominee · validation and independent synthetic reserve",
    xlabel="GBP per observation",
    ylabel="Financial metric",
)
currency(ax.xaxis)
finish()
fig, ax = plt.subplots(figsize=(8, 3.1))
for index, (label, colour, key) in enumerate(
    (
        ("Validation", blue, "validation_mae_interval"),
        ("Independent synthetic reserve", green, "reserve_mae_interval"),
    )
):
    lower, upper = report[key]
    ax.hlines(index, lower, upper, color=colour, linewidth=3)
    ax.scatter(scopes.at[label, "MAE"], index, color=colour, s=44)
    ax.annotate(
        f"£{lower:.2f}–£{upper:.2f}",
        (upper, index),
        xytext=(7, 0),
        textcoords="offset points",
        va="center",
    )
ax.set(
    title="Conditional account-bootstrap MAE intervals",
    yticks=[0, 1],
    yticklabels=scopes.index,
    xlabel="MAE, GBP",
    xlim=(0, max(report["validation_mae_interval"] + report["reserve_mae_interval"]) * 1.42),
)
ax.invert_yaxis()
currency(ax.xaxis)
finish()

## 32. Research limitations

The fictional catalogue and billing schedules constrain generalisation.
The target covers recorded gross issuance. Supported credit, recognition,
cash and profitability evidence would require separate financial sources.

## 33. Business interpretation

A lower validation error can support further investigation for billed-value
planning. It cannot demonstrate forecast accuracy on real accounts or
estimate realised business returns.

## 34. Final conclusions

The workflow provides auditable six-month outcome maturity, GBP losses and
independent synthetic evidence under the approved protocol. The results
remain experimental. The Project Owner accepted Phase 07 engineering with documented limitations.

Boosting's low training MAE and higher validation MAE are consistent with
overfitting or calendar and customer-population changes. The experiment cannot
separate these explanations. Annual and high-value errors warrant separate
attention before using a forecast as a billed-exposure planning input.

**Verified evidence.** Boosting led on validation MAE under the locked rule.
The median baseline's exact matches coexist with substantial missed value away
from the dominant amount. Positive aggregate bias does not exclude high-value
underforecasting. Error concentration and cadence strata matter to exposure planning.

**Methodological strengths and limitations.** Customer group isolation,
verified target maturity and a separate synthetic reserve support transparent
experimental evaluation. Repeated observations, monetary ties and a single
later reserve date limit statistical and commercial inference. The experiment
cannot separate catalogue, billing-timing and overfitting effects.

**Deferred implementation and deployment evidence.** A contractual comparator
requires approved renewal and retention evidence. Real-client evaluation would
require verified billing completeness, adjustment treatment, a broader range
of out-of-time cohorts and error tolerances defined for a concrete billed-value
planning use case. Customer prioritisation and commercial deployment remain
research questions. Forecasts do not establish cash, margin or total lifetime value.
